### Preprocessing of evaluation datasets

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

from dual_ifm.utils import datasets

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

In [ ]:
def check_image_exists(df, dataset_folder):
    # Check for all files existance
    for filename, idx in tqdm(zip(df['image_path'], df.index.to_numpy())):
        if not os.path.exists(
            os.path.join(dataset_dir, dataset_folder, 'dim_512', filename)
        ):
            df.drop([idx], inplace=True)
            print('Dropped: ', filename)
    return df

dataset_dir = './data'

### EyePACS

In [ ]:
labels_file = os.path.join(dataset_dir, 'eyepacs', 'metadata', 'metadata.csv')
df = pd.read_csv(labels_file)

### AREDS

In [ ]:
labels_file = os.path.join(dataset_dir, 'areds', 'metadata', 'metadata.csv')
df = pd.read_csv(labels_file)
df['amd_grouped'] = pd.cut(df['diagnosis_amd_grade'], bins=[0, 3, 9, 12], labels=[0, 1, 2]).astype(int)

In [ ]:
df.head()

In [ ]:
df.to_csv(labels_file, index=False)

### IDRID

In [ ]:
# Clean up label files
labels_file = os.path.join(
    dataset_dir, 'idrid', 'metadata', 'training_label.csv'
)
df_train = pd.read_csv(labels_file)
df_train = df_train.rename(columns={'Risk of macular edema ': 'Risk of macular edema'})
df_train = df_train[['Image name', 'Retinopathy grade', 'Risk of macular edema']]
df_train['age'] = np.nan
df_train['image_path'] = df_train['Image name'].map(
    lambda x: os.path.join('Train_set', x + '.jpg')
)
df_train.to_csv(labels_file, index=False)

labels_file = os.path.join(dataset_dir, 'idrid', 'metadata', 'test_label.csv')
df_test = pd.read_csv(labels_file)
df_test = df_test.rename(columns={'Risk of macular edema ': 'Risk of macular edema'})
df_test = df_test[['Image name', 'Retinopathy grade', 'Risk of macular edema']]
df_test['age'] = np.nan
df_test['image_path'] = df_test['Image name'].map(
    lambda x: os.path.join('Test_set', x + '.jpg')
)
df_test.to_csv(labels_file, index=False)

In [ ]:
# Join label files into a single metadata file
labels_file = os.path.join(
    dataset_dir, 'idrid', 'metadata', 'training_label.csv'
)
df_train = pd.read_csv(labels_file)

labels_file = os.path.join(dataset_dir, 'idrid', 'metadata', 'test_label.csv')
df_test = pd.read_csv(labels_file)

df = pd.concat((df_train, df_test), axis=0)
df['split'] = df['image_path'].map(lambda x: 'test' if 'test' in x.lower() else 'train')
df['patient_id'] = np.arange(df.shape[0])

labels_file = os.path.join(dataset_dir, 'idrid', 'metadata', 'metadata.csv')
df.to_csv(labels_file, index=False)

In [ ]:
# Add binary
labels_file = os.path.join(dataset_dir, 'idrid', 'metadata', 'metadata.csv')
df = pd.read_csv(labels_file)
df['drbin'] = df['Retinopathy grade'].map(lambda x: 1 if x >= 1 else 0)

df.to_csv(labels_file, index=False)

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'idrid', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='Retinopathy grade')

In [ ]:
# Load dataset
dataset, mapping = datasets.load_dataset(
    dataset_dir=dataset_dir,
    dataset_name='idrid',
    image_size=(256, 256),
    feature_name='dr',
    sample_size=None,
    split='train',
    kfold=0,
)
print(len(dataset), mapping)

In [ ]:
df['Retinopathy grade'].value_counts(dropna=False)

### APTOS

In [ ]:
labels_file = os.path.join(dataset_dir, 'aptos', 'metadata', 'metadata.csv')
df = pd.read_csv(labels_file)
df['patient_id'] = np.arange(df.shape[0])
df['image_path'] = df['id_code'].map(lambda x: x + '.jpg')
df['age'] = np.nan
df.to_csv(labels_file, index=False)

In [ ]:
labels_file = os.path.join(dataset_dir, 'aptos', 'metadata', 'metadata.csv')
df = pd.read_csv(labels_file)
df.head()

In [ ]:
df['diagnosis'].value_counts(dropna=False)

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'aptos', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='diagnosis')

In [ ]:
# Load dataset
dataset, mapping = datasets.load_dataset(
    'aptos',
    image_size=(256, 256),
    feature_name='dr',
    sample_size=None,
    split='val',
    kfold=1,
)
print(len(dataset), mapping)

### MESSIDOR-2

In [ ]:
labels_file = os.path.join(
    dataset_dir, 'messidor', 'metadata', 'messidor_data.csv'
)
df = pd.read_csv(labels_file)
df['patient_id'] = np.arange(df.shape[0])
df['image_path'] = df['image_id'].map(lambda x: x.split(sep='.')[0] + '.jpg')
df['age'] = np.nan
df = df[df['adjudicated_gradable'] == 1]
print(df.shape)

In [ ]:
labels_file = os.path.join(
    dataset_dir, 'messidor', 'metadata', 'metadata.csv'
)
df.to_csv(labels_file, index=False)

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'messidor', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='adjudicated_dr_grade',)

In [ ]:
# Load dataset
dataset, mapping = datasets.load_dataset(
    'messidor',
    image_size=(256, 256),
    feature_name='dr',
    sample_size=None,
    split='test',
    kfold=0,
)
print(len(dataset), mapping)

In [ ]:
df['adjudicated_dr_grade'].value_counts(dropna=False)

### DeepDRiD

In [ ]:
def preprocess_deepdrid(df, split='train'):
    if split in ['train', 'val']:
        df = df.rename(
            columns={'left_eye_DR_Level': 'left dr', 'right_eye_DR_Level': 'right dr'}
        )
        df['dr'] = df['right dr'].combine_first(df['left dr'])
    else:
        df = df.rename(columns={'DR_Levels': 'dr'})
        df = df.drop(columns=['Unnamed: 0'])
        df['patient_id'] = df['image_id'].map(lambda x: x.split(sep='_')[0])

    df['age'] = np.nan
    df['split'] = split
    df['image_path'] = df.apply(
        lambda row: os.path.join(
            row['split'], str(row['patient_id']), row['image_id'] + '.jpg'
        ),
        axis=1,
    )
    return df


labels_file = os.path.join(
    dataset_dir, 'deepdrid', 'metadata', 'regular-fundus-training.csv'
)
df_train = pd.read_csv(labels_file)
df_train = preprocess_deepdrid(df_train, 'train')

labels_file = os.path.join(
    dataset_dir, 'deepdrid', 'metadata', 'regular-fundus-validation.csv'
)
df_val = pd.read_csv(labels_file)
df_val = preprocess_deepdrid(df_val, 'val')

labels_file = os.path.join(dataset_dir, 'deepdrid', 'metadata', 'test.csv')
df_test = pd.read_csv(labels_file)
df_test = preprocess_deepdrid(df_test, 'test')

df = pd.concat((df_train, df_val, df_test), axis=0)
df = df.reset_index(drop=True)
df = check_image_exists(df, 'deepdrid')
labels_file = os.path.join(dataset_dir, 'deepdrid', 'metadata', 'metadata.csv')
df.to_csv(labels_file, index=False)

In [ ]:
labels_file = os.path.join(dataset_dir, 'deepdrid', 'metadata', 'metadata.csv')
df = pd.read_csv(labels_file)
df.head()

In [ ]:
df[df['split'] == 'train'].shape

In [ ]:
df_kfold = df[df['split'].isin(['train', 'val'])]
df_kfold['patient_DR_Level'].value_counts(dropna=False)

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'deepdrid', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='patient_DR_Level')

In [ ]:
dataset, mapping = datasets.load_dataset(
    'deepdrid',
    image_size=(256, 256),
    feature_name='dr',
    sample_size=None,
    split='val',
    kfold=1,
)
print(len(dataset), mapping)

### Glaucoma

In [ ]:
labels_file = os.path.join(dataset_dir, 'glaucoma', 'metadata', 'labels.csv')
df = pd.read_csv(labels_file)
df['patient_id'] = np.arange(df.shape[0])
df['image_path'] = df[['label', 'filename']].apply(
    lambda x: os.path.join(x['label'], x['filename']), axis=1
)
df['label'] = df['label'].map(
    {'normal_control': 0, 'early_glaucoma': 1, 'advanced_glaucoma': 2}
)
df['age'] = np.nan

In [ ]:
labels_file = os.path.join(dataset_dir, 'glaucoma', 'metadata', 'metadata.csv')
df.to_csv(labels_file, index=False)

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'glaucoma', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='label')

In [ ]:
dataset, mapping = datasets.load_dataset(
    'glaucoma',
    image_size=(256, 256),
    feature_name='glaucoma',
    sample_size=None,
    split='test',
    kfold=0,
)
print(len(dataset), mapping)

In [ ]:
df['label'].value_counts(dropna=False)

### Papila

In [ ]:
# Counts from the paper: healthy: 333, glaucoma: 87, suspect: 68
labels_file = os.path.join(dataset_dir, 'papila', 'metadata', 'labels.csv')
df = pd.read_csv(labels_file)
# df = df.rename(columns={'ID': 'patient_id', 'filename': 'image_path'})
# df['Diagnosis'] = df['Diagnosis'].map(lambda x: 0 if (x==0) or (x==2) else 1)
# df = df[df['Diagnosis'] < 2]

In [ ]:
df['Diagnosis'].value_counts(dropna=False)

In [ ]:
labels_file = os.path.join(dataset_dir, 'papila', 'metadata', 'metadata.csv')
df.to_csv(labels_file, index=False)

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'papila', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='Diagnosis')

In [ ]:
dataset, mapping = datasets.load_dataset(
    'papila',
    image_size=(256, 256),
    feature_name='glaucoma',
    sample_size=None,
    split='test',
    kfold=0,
)
print(len(dataset), mapping)

### FIVES

In [ ]:
def preprocess_fives(df, split='train'):
    df['age'] = np.nan
    df['split'] = split
    df['image_path'] = df.apply(
        lambda row: os.path.join(
            row['split'], str(row['Number']) + '_' + row['Disease'] + '.jpg'
        ),
        axis=1,
    )
    return df


labels_file = os.path.join(
    dataset_dir, 'fives', 'metadata', 'Quality_Assessment_train.csv'
)
df_train = pd.read_csv(labels_file)
df_train = preprocess_fives(df_train, split='train')

labels_file = os.path.join(
    dataset_dir, 'fives', 'metadata', 'Quality_Assessment_test.csv'
)
df_test = pd.read_csv(labels_file)
df_test = preprocess_fives(df_test, split='test')

df = pd.concat((df_train, df_test), axis=0)
df = df.reset_index(drop=True)
df = check_image_exists(df, 'fives')
df['patient_id'] = np.arange(df.shape[0])
labels_file = os.path.join(dataset_dir, 'fives', 'metadata', 'metadata.csv')
df.to_csv(labels_file, index=False)

In [ ]:
df.head()

In [ ]:
# Generate splits
labels_file = os.path.join(dataset_dir, 'fives', 'metadata', 'metadata.csv')
datasets.generate_splits(labels_file, stratify_col='Disease')

In [ ]:
dataset, mapping = datasets.load_dataset(
    'fives',
    image_size=(256, 256),
    feature_name='disease',
    sample_size=None,
    split='test',
    kfold=0,
)
print(len(dataset), mapping)

### Mean and sd per dataset

In [ ]:
NORMALIZATION_MEAN = {
    'imagenet': [0.485, 0.456, 0.406],
    'eyepacs': [0.340, 0.215, 0.139],
    'areds': [0.497, 0.279, 0.136],
    'ukb': [0.562, 0.269, 0.096],
    'all': [0.397, 0.233, 0.131],
    'idrid': [0.447, 0.216, 0.069],
    'aptos': [0.376, 0.201, 0.063],
    'messidor': [0.476, 0.221, 0.076],
    'deepdrid': [0.429, 0.263, 0.158],
    'glaucoma': [0.695, 0.418, 0.192],
    'papila': [0.324, 0.111, 0.056],
    'fives': [0.34, 0.157, 0.065],
    'idridncc': [0.478, 0.231, 0.065],
}
NORMALIZATION_SD = {
    'imagenet': [0.229, 0.224, 0.225],
    'eyepacs': [0.268, 0.189, 0.153],
    'areds': [0.346, 0.215, 0.142],
    'ukb': [0.356, 0.184, 0.087],
    'all': [0.309, 0.194, 0.143],
    'idrid': [0.306, 0.163, 0.083],
    'aptos': [0.288, 0.158, 0.078],
    'messidor': [0.297, 0.149, 0.065],
    'deepdrid': [0.309, 0.202, 0.147],
    'glaucoma': [0.17, 0.158, 0.145],
    'papila': [0.247, 0.092, 0.046],
    'fives': [0.243, 0.125, 0.06],
    'idridncc': [0.315, 0.166, 0.087],
}

transform = transforms.ToTensor()
dataset, _ = datasets.load_dataset(
    'fives', transform=transform, image_size=(256, 256), sample_size=None, split='train'
)
dataloader = DataLoader(
    dataset, batch_size=4096, shuffle=False, num_workers=0, drop_last=False
)

In [ ]:
def print_list_with_precision(l):
    formatted_l = [round(x, 3) for x in l]
    return formatted_l


dataset_mean, dataset_sd = datasets.get_dataset_normalization(dataloader, 'cuda:0')
print('Mean: ', print_list_with_precision(dataset_mean))
print('SD: ', print_list_with_precision(dataset_sd))

### Check distributions of the splits from any dataset

In [ ]:
fig, ax = plt.subplots(
    2,
    6,
    figsize=(15, 6),
    sharey=True,
)
ax = np.array(ax)

dataset_name = 'idrid'
feature_name = 'dr'
bins = 5

dataset, mapping = datasets.load_dataset(
    dataset_dir=dataset_dir,
    dataset_name=dataset_name,
    transform=None,
    image_size=(256, 256),
    feature_name=feature_name,
    drop_nan=True,
    sample_size=None,
    split='test',
    kfold=0,
)
# mapping = {0: 'no', 1: 'yes'}
df = pd.DataFrame({feature_name: dataset.labels})
df[feature_name] = df[feature_name].map(mapping)
sns.histplot(
    data=df, x=feature_name, bins=bins, stat='percent', shrink=0.6, ax=ax[0, 0]
)
ax[0, 0].bar_label(ax[0, 0].containers[0], fmt='%.3f')
ax[0, 0].set_title('split = test')

for fold in range(1, 6):
    for s, split in enumerate(['train', 'val']):
        dataset, mapping = datasets.load_dataset(
            dataset_dir=dataset_dir,
            dataset_name=dataset_name,
            transform=None,
            image_size=(256, 256),
            feature_name=feature_name,
            drop_nan=True,
            sample_size=None,
            split=split,
            kfold=fold,
        )
        df = pd.DataFrame({feature_name: dataset.labels})
        df[feature_name] = df[feature_name].map(mapping)
        sns.histplot(
            data=df,
            x=feature_name,
            bins=bins,
            stat='percent',
            shrink=0.6,
            ax=ax[s, fold],
        )
        ax[s, fold].bar_label(ax[s, fold].containers[0], fmt='%.3f')
        ax[s, fold].set_title(f'fold = {fold}, split = {split}')

fig.suptitle(f'Distribution of the 5-folds ({dataset_name}, {feature_name})')
plt.tight_layout()